# MCP Architecture Overview

This notebook provides a comprehensive overview of the Model Context Protocol (MCP) architecture and how it enables AI agents to interact with Algorand blockchain services.

In [1]:
import sys
sys.path.append('../00_setup')
from helpers import call_api, check_health
from config import READER_URL, WRITER_URL, MARKET_URL

## What is MCP?

The Model Context Protocol (MCP) is a standardized way for AI agents to interact with external services and data sources. In our Algorand showcase, MCP enables AI agents to:

- **Read** blockchain data (accounts, transactions, assets)
- **Write** transactions (payments, asset transfers)
- **Access** market data and pricing information

## Architecture Components

In [2]:
print("🏗️  MCP Architecture Components")
print("=" * 40)
print()
print("📊 Data Flow:")
print("  AI Agent → MCP Service → Algorand Network")
print()
print("🔧 Service Types:")
print("  📖 Reader MCP  - Blockchain data access")
print("  ✏️  Writer MCP  - Transaction building")
print("  📈 Market MCP  - Price feeds and market data")
print()
print("🌐 Network Integration:")
print("  🧪 Testnet - Safe testing environment")
print("  🚀 Mainnet - Production blockchain (future)")
print()
print("🛡️  Security Features:")
print("  🔒 Read-only operations for Reader")
print("  🔑 Unsigned transactions from Writer")
print("  🧪 Mock fallbacks for testing")

🏗️  MCP Architecture Components

📊 Data Flow:
  AI Agent → MCP Service → Algorand Network

🔧 Service Types:
  📖 Reader MCP  - Blockchain data access
  ✏️  Writer MCP  - Transaction building
  📈 Market MCP  - Price feeds and market data

🌐 Network Integration:
  🧪 Testnet - Safe testing environment
  🚀 Mainnet - Production blockchain (future)

🛡️  Security Features:
  🔒 Read-only operations for Reader
  🔑 Unsigned transactions from Writer
  🧪 Mock fallbacks for testing


## Service Discovery and Health

Let's discover and analyze our MCP services:

In [3]:
import asyncio
import time

async def analyze_service(name, url, description):
    """Analyze a single MCP service"""
    print(f"\n🔍 Analyzing {name}")
    print(f"   URL: {url}")
    print(f"   Purpose: {description}")
    
    # Health check with timing
    start_time = time.time()
    health_result = await call_api(f"{url}/health")
    response_time = (time.time() - start_time) * 1000
    
    if health_result.get("status") == "ok":
        print(f"   Status: ✅ Healthy ({response_time:.1f}ms)")
        
        # Try to get service capabilities
        tools_result = await call_api(f"{url}/tools/list")
        if tools_result.get("tools"):
            tools = tools_result["tools"]
            print(f"   Tools: {len(tools)} available")
            for tool in tools[:3]:  # Show first 3 tools
                print(f"     • {tool.get('name', 'Unknown')}: {tool.get('description', 'No description')}")
            if len(tools) > 3:
                print(f"     ... and {len(tools) - 3} more")
        else:
            print(f"   Tools: Unable to retrieve tool list")
            
    elif health_result.get("message") == "Mock service healthy":
        print(f"   Status: 🧪 Mock mode ({response_time:.1f}ms)")
    else:
        print(f"   Status: ❌ Unavailable ({response_time:.1f}ms)")
        print(f"   Error: {health_result.get('error', 'Service not responding')}")
    
    return health_result.get("status") == "ok"

# Analyze all services
services = [
    ("Reader MCP", READER_URL, "Access blockchain data (accounts, assets, transactions)"),
    ("Writer MCP", WRITER_URL, "Build unsigned transactions for signing and submission"),
    ("Market MCP", MARKET_URL, "Provide market data and price feeds")
]

print("🔍 MCP Service Discovery")
print("=" * 30)

healthy_services = 0
for name, url, description in services:
    is_healthy = await analyze_service(name, url, description)
    if is_healthy:
        healthy_services += 1

print(f"\n📊 Service Summary: {healthy_services}/{len(services)} services operational")

🔍 MCP Service Discovery

🔍 Analyzing Reader MCP
   URL: http://localhost:3500
   Purpose: Access blockchain data (accounts, assets, transactions)
   Status: ✅ Healthy (5.7ms)
   Tools: 5 available
     • get_account_info: Get account information including balance and assets
     • get_transaction: Get transaction details by ID
     • get_asset_info: Get asset information
     ... and 2 more

🔍 Analyzing Writer MCP
   URL: http://localhost:8788
   Purpose: Build unsigned transactions for signing and submission
   Status: ✅ Healthy (1.9ms)
   Tools: 3 available
     • build_payment_transaction: Build a real Algorand payment transaction
     • build_asset_transfer: Build a real asset transfer transaction
     • get_network_status: Get current Algorand testnet status

🔍 Analyzing Market MCP
   URL: http://localhost:8789
   Purpose: Provide market data and price feeds
🧪 Using mock data for Market MCP: health
   Status: ✅ Healthy (0.0ms)
🧪 Using mock data for Market MCP: list
   Tools: 3 ava

## MCP Protocol Features

Let's demonstrate key MCP protocol features:

In [4]:
print("⚡ MCP Protocol Features")
print("=" * 30)

# 1. Tool Discovery
print("\n1. 🔍 Tool Discovery")
print("   Standard endpoint: /tools/list")
print("   Returns: Available tools and their descriptions")

reader_tools = await call_api(f"{READER_URL}/tools/list")
if reader_tools.get("tools"):
    print(f"   Reader tools found: {len(reader_tools['tools'])}")
else:
    print("   Reader tools: Using mock data")

# 2. Standardized Responses
print("\n2. 📝 Standardized Response Format")
print("   All responses include:")
print("     • success: boolean")
print("     • data: response payload")
print("     • error: error message (if applicable)")

# 3. Health Monitoring
print("\n3. 🏥 Health Monitoring")
print("   Standard endpoint: /health")
print("   Provides: Service status and metadata")

# 4. Error Handling
print("\n4. 🛡️  Error Handling")
print("   Features:")
print("     • Graceful degradation")
print("     • Mock fallbacks")
print("     • Clear error messages")
print("     • Retry mechanisms")

# 5. Security
print("\n5. 🔒 Security Model")
print("   Reader: Read-only access, no sensitive operations")
print("   Writer: Builds unsigned transactions only")
print("   Market: Public data, no authentication required")
print("   Network: Testnet operations for safe development")

⚡ MCP Protocol Features

1. 🔍 Tool Discovery
   Standard endpoint: /tools/list
   Returns: Available tools and their descriptions
   Reader tools found: 5

2. 📝 Standardized Response Format
   All responses include:
     • success: boolean
     • data: response payload
     • error: error message (if applicable)

3. 🏥 Health Monitoring
   Standard endpoint: /health
   Provides: Service status and metadata

4. 🛡️  Error Handling
   Features:
     • Graceful degradation
     • Mock fallbacks
     • Clear error messages
     • Retry mechanisms

5. 🔒 Security Model
   Reader: Read-only access, no sensitive operations
   Writer: Builds unsigned transactions only
   Market: Public data, no authentication required
   Network: Testnet operations for safe development


## AI Agent Integration

Let's demonstrate how AI agents would interact with our MCP services:

In [5]:
print("🤖 AI Agent Integration Patterns")
print("=" * 40)

# Simulate an AI agent workflow
print("\n📋 Example AI Agent Workflow:")
print("1. Agent receives user request: 'Send 1 ALGO to Bob'")
print("2. Agent calls Reader MCP to check sender balance")
print("3. Agent calls Writer MCP to build transaction")
print("4. Agent presents transaction to user for signing")
print("5. Agent submits signed transaction (future feature)")

# Demonstrate actual calls
print("\n🔄 Simulating Agent Workflow:")

# Step 1: Check balance
alice_address = "7ZUECA7HFLZTXENRV24SHLU4AVPUTMTTDUFUBNBD64C73F3UHRTHAIOF6Q"
bob_address = "GD64YIY3TWGDMCNPP553DZPPR6LDUSFQOIJVFDPPXWEG3FVOJCCDBBHU5A"

print("   Step 1: Checking Alice's balance...")
balance_result = await call_api(f"{READER_URL}/tools/get_account_info", {"address": alice_address})

if balance_result.get("success"):
    balance = balance_result["account"]["amount"]
    balance_algo = balance / 1_000_000
    print(f"   ✅ Alice's balance: {balance_algo:.6f} ALGO")
    
    # Step 2: Build transaction
    if balance >= 1_001_000:  # 1 ALGO + fee
        print("   Step 2: Building payment transaction...")
        
        tx_result = await call_api(f"{WRITER_URL}/tools/build_payment_transaction", {
            "fromAddress": alice_address,
            "toAddress": bob_address,
            "microAlgos": 1_000_000,  # 1 ALGO
            "note": "AI Agent payment"
        })
        
        if tx_result.get("success"):
            print(f"   ✅ Transaction built: {tx_result.get('txId')}")
            print(f"   📝 Fee: {tx_result.get('fee', 0)} microAlgos")
            print("   🎯 Ready for user signing and submission")
        else:
            print(f"   ❌ Transaction build failed: {tx_result.get('error')}")
    else:
        print("   ❌ Insufficient balance for transaction")
else:
    print(f"   ❌ Balance check failed: {balance_result.get('error')}")
    print("   🧪 Falling back to mock workflow demonstration")

🤖 AI Agent Integration Patterns

📋 Example AI Agent Workflow:
1. Agent receives user request: 'Send 1 ALGO to Bob'
2. Agent calls Reader MCP to check sender balance
3. Agent calls Writer MCP to build transaction
4. Agent presents transaction to user for signing
5. Agent submits signed transaction (future feature)

🔄 Simulating Agent Workflow:
   Step 1: Checking Alice's balance...
   ✅ Alice's balance: 703.754899 ALGO
   Step 2: Building payment transaction...
   ✅ Transaction built: 4OW2EDDH2DLIDMRSXNXMPYF5XNLSK3POKVHGJ2CCCQKRV5MU7SDA
   📝 Fee: 1000 microAlgos
   🎯 Ready for user signing and submission


## Advanced MCP Features

Let's explore advanced features and capabilities:

In [6]:
print("🚀 Advanced MCP Features")
print("=" * 30)

# 1. Service Composition
print("\n1. 🔗 Service Composition")
print("   Agents can chain multiple MCP calls:")
print("     Reader → Writer → Market (price check)")
print("     Multiple Writers for batch transactions")
print("     Error handling across service boundaries")

# 2. Context Preservation
print("\n2. 💾 Context Preservation")
print("   Transaction context maintained across calls:")
print("     • User preferences and settings")
print("     • Transaction history and patterns")
print("     • Error recovery strategies")

# 3. Real-time Updates
print("\n3. ⚡ Real-time Capabilities")
print("   Future enhancements:")
print("     • WebSocket connections for live data")
print("     • Transaction status monitoring")
print("     • Block confirmation tracking")

# 4. Extensibility
print("\n4. 🔧 Extensibility")
print("   Easy to add new services:")
print("     • DeFi protocol integration")
print("     • NFT marketplace access")
print("     • Cross-chain bridge support")

# 5. Analytics
print("\n5. 📊 Analytics and Monitoring")
print("   Built-in observability:")
print("     • Request/response logging")
print("     • Performance metrics")
print("     • Error rate tracking")
print("     • Usage pattern analysis")

# Current implementation status
print("\n📈 Current Implementation Status:")
features = [
    ("✅", "Basic MCP protocol"),
    ("✅", "Reader service (blockchain data)"),
    ("✅", "Writer service (payment transactions)"),
    ("✅", "Error handling and fallbacks"),
    ("✅", "Health monitoring"),
    ("🔄", "Market data service"),
    ("🔄", "Asset transfer transactions"),
    ("📋", "Smart contract interactions"),
    ("📋", "Real-time WebSocket updates"),
    ("📋", "Cross-chain capabilities")
]

for status, feature in features:
    print(f"   {status} {feature}")

print("\n📌 Legend: ✅ Complete | 🔄 In Progress | 📋 Planned")

🚀 Advanced MCP Features

1. 🔗 Service Composition
   Agents can chain multiple MCP calls:
     Reader → Writer → Market (price check)
     Multiple Writers for batch transactions
     Error handling across service boundaries

2. 💾 Context Preservation
   Transaction context maintained across calls:
     • User preferences and settings
     • Transaction history and patterns
     • Error recovery strategies

3. ⚡ Real-time Capabilities
   Future enhancements:
     • WebSocket connections for live data
     • Transaction status monitoring
     • Block confirmation tracking

4. 🔧 Extensibility
   Easy to add new services:
     • DeFi protocol integration
     • NFT marketplace access
     • Cross-chain bridge support

5. 📊 Analytics and Monitoring
   Built-in observability:
     • Request/response logging
     • Performance metrics
     • Error rate tracking
     • Usage pattern analysis

📈 Current Implementation Status:
   ✅ Basic MCP protocol
   ✅ Reader service (blockchain data)
   ✅ W

## Performance and Scalability

Let's analyze the performance characteristics of our MCP services:

In [7]:
import asyncio
import time
import statistics

async def performance_test(service_name, url, endpoint, data, iterations=5):
    """Test service performance"""
    print(f"\n⚡ Testing {service_name} performance...")
    response_times = []
    
    for i in range(iterations):
        start_time = time.time()
        result = await call_api(f"{url}{endpoint}", data)
        response_time = (time.time() - start_time) * 1000
        response_times.append(response_time)
        
        status = "✅" if result.get("success") else "❌"
        print(f"   Test {i+1}: {status} {response_time:.1f}ms")
    
    # Calculate statistics
    avg_time = statistics.mean(response_times)
    min_time = min(response_times)
    max_time = max(response_times)
    
    print(f"   📊 Average: {avg_time:.1f}ms")
    print(f"   ⚡ Fastest: {min_time:.1f}ms")
    print(f"   🐌 Slowest: {max_time:.1f}ms")
    
    return avg_time

print("🔬 Performance Analysis")
print("=" * 25)

# Test Reader service
reader_perf = await performance_test(
    "Reader MCP",
    READER_URL,
    "/tools/get_account_info",
    {"address": alice_address}
)

# Test Writer service
writer_perf = await performance_test(
    "Writer MCP",
    WRITER_URL,
    "/tools/build_payment_transaction",
    {
        "fromAddress": alice_address,
        "toAddress": bob_address,
        "microAlgos": 1000000,
        "note": "Performance test"
    }
)

# Performance summary
print(f"\n📈 Performance Summary:")
print(f"   Reader Average: {reader_perf:.1f}ms")
print(f"   Writer Average: {writer_perf:.1f}ms")

if reader_perf < 100 and writer_perf < 200:
    print("   🚀 Excellent performance - suitable for real-time applications")
elif reader_perf < 500 and writer_perf < 1000:
    print("   ✅ Good performance - suitable for interactive applications")
else:
    print("   ⚠️  Performance needs optimization for production use")

print("\n💡 Performance Tips:")
print("   • Reader operations are typically faster (cached data)")
print("   • Writer operations involve network calls to Algorand")
print("   • Mock mode provides instant responses for testing")
print("   • Connection pooling improves sustained performance")

🔬 Performance Analysis

⚡ Testing Reader MCP performance...
   Test 1: ✅ 191.0ms
   Test 2: ✅ 124.2ms
   Test 3: ✅ 128.4ms
   Test 4: ✅ 120.9ms
   Test 5: ✅ 121.3ms
   📊 Average: 137.2ms
   ⚡ Fastest: 120.9ms
   🐌 Slowest: 191.0ms

⚡ Testing Writer MCP performance...
   Test 1: ✅ 136.9ms
   Test 2: ✅ 97.7ms
   Test 3: ✅ 100.6ms
   Test 4: ✅ 97.4ms
   Test 5: ✅ 95.8ms
   📊 Average: 105.7ms
   ⚡ Fastest: 95.8ms
   🐌 Slowest: 136.9ms

📈 Performance Summary:
   Reader Average: 137.2ms
   Writer Average: 105.7ms
   ✅ Good performance - suitable for interactive applications

💡 Performance Tips:
   • Reader operations are typically faster (cached data)
   • Writer operations involve network calls to Algorand
   • Mock mode provides instant responses for testing
   • Connection pooling improves sustained performance


## Architecture Summary

Let's summarize the MCP architecture and its benefits:

In [8]:
print("📋 MCP Architecture Summary")
print("=" * 35)

print("\n🎯 Key Benefits:")
print("   ✅ Standardized AI agent integration")
print("   ✅ Modular and composable services")
print("   ✅ Robust error handling and fallbacks")
print("   ✅ Security through separation of concerns")
print("   ✅ Easy testing with mock services")
print("   ✅ Scalable and performant design")

print("\n🏗️  Architecture Principles:")
print("   • Single Responsibility: Each service has one focus")
print("   • Loose Coupling: Services are independent")
print("   • High Cohesion: Related functions grouped together")
print("   • Fail-Safe Design: Graceful degradation")
print("   • Observable: Comprehensive monitoring")

print("\n🔮 Future Roadmap:")
print("   📋 Smart contract interaction service")
print("   📋 DeFi protocol integration")
print("   📋 NFT marketplace service")
print("   📋 Cross-chain bridge support")
print("   📋 Real-time event streaming")
print("   📋 Advanced analytics service")

print("\n🎉 Ready for AI Agent Development!")
print("   The MCP architecture provides a solid foundation")
print("   for building sophisticated Algorand-powered AI agents.")
print("   Start with the Reader and Writer services,")
print("   then expand with additional capabilities as needed.")

📋 MCP Architecture Summary

🎯 Key Benefits:
   ✅ Standardized AI agent integration
   ✅ Modular and composable services
   ✅ Robust error handling and fallbacks
   ✅ Security through separation of concerns
   ✅ Easy testing with mock services
   ✅ Scalable and performant design

🏗️  Architecture Principles:
   • Single Responsibility: Each service has one focus
   • Loose Coupling: Services are independent
   • High Cohesion: Related functions grouped together
   • Fail-Safe Design: Graceful degradation
   • Observable: Comprehensive monitoring

🔮 Future Roadmap:
   📋 Smart contract interaction service
   📋 DeFi protocol integration
   📋 NFT marketplace service
   📋 Cross-chain bridge support
   📋 Real-time event streaming
   📋 Advanced analytics service

🎉 Ready for AI Agent Development!
   The MCP architecture provides a solid foundation
   for building sophisticated Algorand-powered AI agents.
   Start with the Reader and Writer services,
   then expand with additional capabilities 